# 07a Geostatistics: data preparation and QC

**Manuscript outputs:** Supporting QC for Section 4.2 and Method S1, B3 to B4 (placeholder audit, 47-cell table, spatial weights)

**Inputs:** completed attribution table (stage 06), DSAS rates layer, GLORYS midpoint GeoPackage

Inputs are read from `Master_raw/`; everything produced is written to `Outputs/` (manuscript figures in `Outputs/Figures`, tables in `Outputs/Tables`). The first code cell lists every path. Run the cells in order.

# 01 | Geostatistics data preparation and QC

Reproducible preparation of transect and native GLORYS-cell inputs. Run from a fresh kernel in order. This notebook requires the original local data files; it has not been executed in this environment. The three original DSAS placeholder records are retained in original fields but set missing in regression analysis fields. IQR observations are flagged, not removed. Spatial weights are exported with an explicit Pixel_ID order; neither Queen nor Rook is declared scientifically preferred.

## Stage 1 | Paths, input schema and baseline audit

In [1]:
from pathlib import Path

# ------------------------------------------------------------
# REPOSITORY ROOT: first folder above this notebook that contains Master_raw
# (e.g. D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024)
# ------------------------------------------------------------
_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents) if (p / "Master_raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No Master_raw folder found above {_here}. Open this notebook inside the repository.")
RAW = ROOT / "Master_raw"          # inputs
OUTPUTS = ROOT / "Outputs"         # everything this repository produces
FIGURES = OUTPUTS / "Figures"      # manuscript figures
TABLES = OUTPUTS / "Tables"        # manuscript tables
TEXT_STATS = TABLES / "Text_statistics"
for _d in (FIGURES, TABLES, TEXT_STATS):
    _d.mkdir(parents=True, exist_ok=True)
OVERWRITE = True                   # re-running replaces this notebook's earlier outputs


def first_existing(*candidates):
    """Return the first path that exists (otherwise the first one)."""
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return Path(candidates[0])


def save_table(df, name, index=False, folder=None):
    path = (folder or TABLES) / f"{name}.csv"
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("Saved:", path.relative_to(ROOT))


def save_figure(fig, name, dpi=600):
    for ext in ("jpg",):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=dpi, bbox_inches="tight", pil_kwargs=({"quality": 95} if ext == "jpg" else None))
    print("Saved:", (FIGURES / f"{name}.jpg").relative_to(ROOT))


def remove_existing(*files):
    for f in map(Path, files):
        for t in (list(f.parent.glob(f.stem + ".*")) if f.suffix == ".shp" else [f]):
            if t.exists():
                if not OVERWRITE:
                    raise FileExistsError(t)
                t.unlink()

# ------------------------------------------------------------
# INPUTS AND OUTPUT FOLDERS FOR THIS NOTEBOOK
# ------------------------------------------------------------
TRANSECT_RATES_DIR = RAW / "Transect_rates"
DSAS_RATES_GPKG = TRANSECT_RATES_DIR / "ke_30mtr_transects_dsas_uncy.gpkg"
DSAS_RATES_LAYER = "ke_30mtr_transect_dsas_uncy_final_rates_20260915_112207"
GLORYS_DIR = OUTPUTS / "GLORYS_attribution"
GLORYS_MIDPOINT_GPKG = GLORYS_DIR / "ke_transects_GLORYS_midpoint_attributed.gpkg"
COMPLETE_LAYER = "transects_complete"
COMPLETE_GPKG = first_existing(GLORYS_DIR / "ke_transects_GLORYS_complete_attribution_zoned.gpkg",
                               TRANSECT_RATES_DIR / "ke_transects_GLORYS_complete_attribution_zoned.gpkg")
COMPLETE_CSV = GLORYS_DIR / "ke_transects_GLORYS_complete_analysis_input_zoned.csv"
GEOSTATS_DIR = OUTPUTS / "Geostatistics"
GEOSTATS_TRANSECT_DIR = GEOSTATS_DIR / "Transect_Level"
GEOSTATS_CELL_DIR = GEOSTATS_DIR / "GLORYS_Cell_Level"

# The QC table is the stage 06 CSV. If only the GeoPackage is available
# (e.g. the copy in Master_raw/Transect_rates), write the same table from it.
if not COMPLETE_CSV.exists() and COMPLETE_GPKG.exists():
    import geopandas as _gpd
    _g = _gpd.read_file(COMPLETE_GPKG, layer=COMPLETE_LAYER)
    COMPLETE_CSV.parent.mkdir(parents=True, exist_ok=True)
    _g.drop(columns="geometry").to_csv(COMPLETE_CSV, index=False, encoding="utf-8-sig", na_rep="")
    print("Built", COMPLETE_CSV.relative_to(ROOT), "from", COMPLETE_GPKG.relative_to(ROOT))

print("Repository root:", ROOT)
for _label, _p in [("COMPLETE_CSV", COMPLETE_CSV), ("DSAS_RATES_GPKG", DSAS_RATES_GPKG), ("GLORYS_MIDPOINT_GPKG", GLORYS_MIDPOINT_GPKG)]:
    print(f"  {'ok     ' if _p.exists() else 'MISSING'} {_label}: {_p.relative_to(ROOT)}")

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
SOURCE = COMPLETE_CSV
OUT = GEOSTATS_DIR
TRANSECT_DIR = GEOSTATS_TRANSECT_DIR
CELL_DIR = GEOSTATS_CELL_DIR
for directory in (TRANSECT_DIR, CELL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
raw = pd.read_csv(SOURCE)
raw.columns = raw.columns.str.strip()
required = ["Transect_ID", "Pixel_ID", "Zone", "WLR", "WCI", "WSE", "SL_Trend_mm_yr", "SL_Variability_m", "Longitude", "Latitude"]
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f"Required input fields missing: {missing}")
summary = pd.Series({"Records": len(raw), "Unique transects": raw.Transect_ID.nunique(), "Duplicate IDs": raw.Transect_ID.duplicated().sum(), "GLORYS cells": raw.Pixel_ID.nunique(), "Missing required values": int(raw[required].isna().sum().sum())})
display(summary.to_frame("Value"))
if len(raw) != 12455 or raw.Transect_ID.nunique() != len(raw) or raw.Pixel_ID.nunique() != 47 or raw[required].isna().any().any():
    raise ValueError("Baseline audit differs from verified input; inspect source before continuing.")
print("Input:", SOURCE)

Repository root: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024
  ok      COMPLETE_CSV: Outputs\GLORYS_attribution\ke_transects_GLORYS_complete_analysis_input_zoned.csv
  ok      DSAS_RATES_GPKG: Master_raw\Transect_rates\ke_30mtr_transects_dsas_uncy.gpkg
  ok      GLORYS_MIDPOINT_GPKG: Outputs\GLORYS_attribution\ke_transects_GLORYS_midpoint_attributed.gpkg


,Value
Records,12455
Unique transects,12455
Duplicate IDs,0
GLORYS cells,47
Missing required values,0


Input: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\GLORYS_attribution\ke_transects_GLORYS_complete_analysis_input_zoned.csv


## Stage 2 | Numeric checks and preliminary IQR flags (no exclusions)

In [2]:
transect = raw.copy()
for name in ["WLR", "WCI", "WSE", "SL_Trend_mm_yr", "SL_Variability_m", "Longitude", "Latitude"]:
    transect[name] = pd.to_numeric(transect[name], errors="coerce")
if transect[["WLR", "WCI", "WSE", "SL_Trend_mm_yr", "SL_Variability_m", "Longitude", "Latitude"]].isna().any().any():
    raise ValueError("Non-numeric values detected in required numeric fields.")
if not (transect.Longitude.between(-180, 180).all() and transect.Latitude.between(-90, 90).all()):
    raise ValueError("Invalid longitude/latitude ranges.")
q1, q3 = transect.WLR.quantile([.25, .75]); iqr = q3-q1
transect["WLR_Outlier_IQR_Raw"] = ~transect.WLR.between(q1-1.5*iqr, q3+1.5*iqr)
print(f"Raw IQR limits: {q1-1.5*iqr:.3f}, {q3+1.5*iqr:.3f}; flags: {transect.WLR_Outlier_IQR_Raw.sum():,} (no records removed)")

Raw IQR limits: -0.958, 1.089; flags: 2,899 (no records removed)


## Stage 4 | Source DSAS verification and regression QC

The original source has `BaselineID` and `TransOrder`, **not** `Transect_ID`. Identify source records by the regression-value pattern; do not equate row indices or `TransOrder` to the attributed `Transect_ID`. The original source is read only. Verify exactly three source rows have the pattern and `ShrCount = 2` before applying the documented exclusion.

In [3]:
import geopandas as gpd
DSAS_FILE = DSAS_RATES_GPKG
DSAS_LAYER = DSAS_RATES_LAYER
if not DSAS_FILE.is_file():
    raise FileNotFoundError(DSAS_FILE)
source_dsas = gpd.read_file(DSAS_FILE, layer=DSAS_LAYER)
reg_fields = ["LRR","LR2","LCI90","LSE","WLR","WR2","WCI","WSE"]
needed = ["ShrCount", *reg_fields]
if any(c not in source_dsas for c in needed):
    raise ValueError("Original DSAS source is missing required audit fields.")
def is_sentinel(s):
    return np.isclose(pd.to_numeric(s,errors="coerce").to_numpy(dtype=float),9999.99,rtol=0,atol=.001)
source_flag = np.logical_and.reduce([is_sentinel(source_dsas[c]) for c in reg_fields])
source_suspects = source_dsas.loc[source_flag].copy()
display(source_suspects[[c for c in ["BaselineID","TransOrder","ShrCount",*reg_fields,"EPR","NSM","START_DATE","END_DATE"] if c in source_suspects]])
print("Original source records:",len(source_dsas),"; all-eight-regression-field sentinel rows:",len(source_suspects))
if len(source_dsas)!=12455 or len(source_suspects)!=3 or not (pd.to_numeric(source_suspects.ShrCount,errors="coerce")==2).all():
    raise ValueError("Original source verification differs from the established three-row pattern.")
source_suspects.drop(columns="geometry",errors="ignore").to_csv(TRANSECT_DIR/"original_DSAS_regression_placeholder_audit.csv",index=False,encoding="utf-8-sig")

,BaselineID,TransOrder,ShrCount,LRR,LR2,LCI90,LSE,WLR,WR2,WCI,WSE,EPR,NSM,START_DATE,END_DATE
3142,3,3146,2,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,36.001385,38.565868,06/15/2006,07/11/2007
3143,3,3147,2,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,14.094456,29.308746,06/12/2005,07/11/2007
3144,3,3164,2,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,9999.990234,12.626114,101.216461,07/05/2008,07/11/2016


Original source records: 12455 ; all-eight-regression-field sentinel rows: 3


In [4]:
qc = transect.copy()
for field in reg_fields:
    if field in qc:
        qc[field+"_Original"] = qc[field]
if any(c not in qc for c in ["WLR","WCI","WSE"]):
    raise ValueError("Required attributed regression fields missing.")
qc["Invalid_Regression"] = np.logical_and.reduce([is_sentinel(qc[c]) for c in ["WLR","WCI","WSE"]])
if qc.Invalid_Regression.sum()!=3:
    raise ValueError("Attributed source has an unexpected placeholder count.")
qc["QC_Reason"] = np.where(qc.Invalid_Regression,"Original DSAS: regression fields 9999.99; ShrCount=2; unavailable regression result","No verified regression placeholder")
for field in reg_fields:
    if field in qc:
        qc[field+"_Analysis"] = pd.to_numeric(qc[field],errors="coerce")
        qc.loc[qc.Invalid_Regression,field+"_Analysis"] = np.nan
q1,q3 = qc.WLR_Analysis.quantile([.25,.75]); iqr=q3-q1
qc["WLR_Outlier_Analysis"] = qc.WLR_Analysis.notna() & ~qc.WLR_Analysis.between(q1-1.5*iqr,q3+1.5*iqr)
if qc.WLR_Analysis.notna().sum()!=12452:
    raise ValueError("Expected 12,452 valid WLR values.")
qc.to_csv(TRANSECT_DIR/"transect_analysis_QC_verified.csv",index=False,encoding="utf-8-sig")
zone_qc = qc.groupby("Zone").agg(Transects=("Transect_ID","size"),Invalid_Regression=("Invalid_Regression","sum"),Valid_WLR=("WLR_Analysis","count"),IQR_Flags=("WLR_Outlier_Analysis","sum"),WLR_Mean=("WLR_Analysis","mean"),WLR_Median=("WLR_Analysis","median"))
zone_qc["IQR_Flag_Percent"] = 100*zone_qc.IQR_Flags/zone_qc.Valid_WLR
zone_qc.to_csv(TRANSECT_DIR/"WLR_QC_by_Zone.csv",encoding="utf-8-sig")
display(zone_qc.round(3))
display(qc.loc[qc.Invalid_Regression,["Transect_ID","Zone","Pixel_ID","WLR","WLR_Analysis","Longitude","Latitude"]])
print("Valid WLR:",qc.WLR_Analysis.notna().sum(),"; IQR flags retained:",qc.WLR_Outlier_Analysis.sum())

,Transects,Invalid_Regression,Valid_WLR,IQR_Flags,WLR_Mean,WLR_Median,IQR_Flag_Percent
Zone,,,,,,,
Kilifi,5445,3,5442,1344,0.568,0.139,24.697
Kwale-Mombasa,3910,0,3910,764,0.508,0.007,19.540
Tana-Manda,3100,0,3100,788,-0.920,-0.254,25.419


,Transect_ID,Zone,Pixel_ID,WLR,WLR_Analysis,Longitude,Latitude
3142,3143,Kilifi,R024_C014,9999.99,NaN,40.202975,-2.750678
3143,3144,Kilifi,R024_C014,9999.99,NaN,40.202902,-2.750881
3144,3145,Kilifi,R024_C014,9999.99,NaN,40.201589,-2.754418


Valid WLR: 12452 ; IQR flags retained: 2896


## Stage 5 | Rebuild the native GLORYS-cell analytical table

Aggregate only verified `WLR_Analysis` values by `Pixel_ID`. Retain both mean and median as alternative descriptive summaries; do not select the regression response based on significance. Record sample size, missing WLR count, outlier flags and all zones touched by a cell. The coordinate fields below are **mean transect positions for reference only**, not GLORYS grid centres; do not construct cell spatial weights from them. No GLORYS cell is automatically considered statistically independent of adjacent cells.

In [5]:
verified_file = TRANSECT_DIR/"transect_analysis_QC_verified.csv"
verified = pd.read_csv(verified_file)
if len(verified)!=12455 or verified.Transect_ID.nunique()!=12455 or verified.Pixel_ID.nunique()!=47:
    raise ValueError("Verified transect input fails count/ID checks.")
for col in ["WLR_Analysis","SL_Trend_mm_yr","SL_Variability_m","Longitude","Latitude"]:
    verified[col] = pd.to_numeric(verified[col],errors="coerce")
if verified[["SL_Trend_mm_yr","SL_Variability_m","Longitude","Latitude"]].isna().any().any():
    raise ValueError("Missing numeric predictor/coordinate values.")
for col in ["SL_Trend_mm_yr","SL_Variability_m"]:
    spread = verified.groupby("Pixel_ID")[col].agg(lambda v: v.max()-v.min())
    if (spread.abs()>1e-9).any():
        raise ValueError(f"Inconsistent native GLORYS predictor {col} within Pixel_ID.")
# Explicit booleans avoid treating CSV strings 'False' as truthy.
for col in ["Invalid_Regression","WLR_Outlier_Analysis"]:
    values = verified[col].astype(str).str.strip().str.lower()
    if not values.isin(["true","false"]).all():
        raise ValueError(f"Invalid boolean values in {col}")
    verified[col] = values.eq("true")
if verified.Invalid_Regression.sum()!=3 or verified.WLR_Analysis.notna().sum()!=12452:
    raise ValueError("QC totals differ from verified stage 4.")
cell = verified.groupby("Pixel_ID",as_index=False).agg(
    Transect_Count=("Transect_ID","size"),Valid_WLR_Count=("WLR_Analysis","count"),
    Invalid_WLR_Count=("Invalid_Regression","sum"),IQR_Flag_Count=("WLR_Outlier_Analysis","sum"),
    WLR_Mean=("WLR_Analysis","mean"),WLR_Median=("WLR_Analysis","median"),
    WLR_SD=("WLR_Analysis","std"),WLR_Min=("WLR_Analysis","min"),WLR_Max=("WLR_Analysis","max"),
    SL_Trend_mm_yr=("SL_Trend_mm_yr","first"),SL_Variability_m=("SL_Variability_m","first"),
    Transect_Mean_Longitude=("Longitude","mean"),Transect_Mean_Latitude=("Latitude","mean"))
zones = verified.groupby("Pixel_ID")["Zone"].agg(lambda x:" | ".join(sorted(x.dropna().astype(str).unique()))).reset_index(name="Zones")
cell=cell.merge(zones,on="Pixel_ID",validate="one_to_one")
cell["IQR_Flag_Percent"] = 100*cell.IQR_Flag_Count/cell.Valid_WLR_Count
cell["Mean_Median_Difference"] = cell.WLR_Mean-cell.WLR_Median
if not (len(cell)==47 and cell.Pixel_ID.is_unique and cell.Transect_Count.sum()==12455 and cell.Valid_WLR_Count.sum()==12452 and cell.Invalid_WLR_Count.sum()==3 and cell.WLR_Median.notna().all()):
    raise ValueError("Cell aggregation validation failed.")
cell_file=CELL_DIR/"glorys_cell_analysis_QC_verified.csv"
cell.to_csv(cell_file,index=False,encoding="utf-8-sig")
print("TABLE 1 | Verification")
display(pd.Series({"Cells":len(cell),"Transects":int(cell.Transect_Count.sum()),"Valid WLR":int(cell.Valid_WLR_Count.sum()),"Invalid WLR":int(cell.Invalid_WLR_Count.sum()),"Minimum valid per cell":int(cell.Valid_WLR_Count.min()),"Maximum valid per cell":int(cell.Valid_WLR_Count.max())}).to_frame("Value"))
print("TABLE 2 | Cell distribution")
display(cell[["WLR_Mean","WLR_Median","Mean_Median_Difference","Valid_WLR_Count"]].describe().round(3))
print("TABLE 3 | Largest mean–median differences")
display(cell.assign(Abs_Difference=cell.Mean_Median_Difference.abs()).sort_values("Abs_Difference",ascending=False)[["Pixel_ID","Zones","Valid_WLR_Count","WLR_Mean","WLR_Median","WLR_Min","WLR_Max","Mean_Median_Difference"]].head(10).round(3))
print("Saved:",cell_file)

TABLE 1 | Verification


,Value
Cells,47
Transects,12455
Valid WLR,12452
Invalid WLR,3
Minimum valid per cell,3
Maximum valid per cell,555


TABLE 2 | Cell distribution


,WLR_Mean,WLR_Median,Mean_Median_Difference,Valid_WLR_Count
count,47.000,47.000,47.000,47.000
mean,0.258,0.018,0.240,264.936
std,2.063,0.921,1.808,123.649
min,-5.866,-2.135,-5.025,3.000
25%,-0.221,-0.180,-0.039,157.000
50%,0.031,0.026,0.005,297.000
75%,0.353,0.195,0.094,338.000
max,9.697,4.383,8.083,555.000


TABLE 3 | Largest mean–median differences


,Pixel_ID,Zones,Valid_WLR_Count,WLR_Mean,WLR_Median,WLR_Min,WLR_Max,Mean_Median_Difference
2,R001_C005,Kwale-Mombasa,236,9.697,1.614,-0.700,102.279,8.083
34,R024_C014,Kilifi,160,5.513,-2.135,-28.786,39.718,7.648
40,R026_C018,Tana-Manda,294,-5.866,-0.842,-92.871,15.654,-5.025
0,R001_C003,Kwale-Mombasa,376,-3.028,-2.101,-46.347,17.886,-0.927
1,R001_C004,Kwale-Mombasa,309,1.562,0.650,-22.144,22.967,0.912
38,R026_C016,Tana-Manda,248,-1.103,-0.424,-6.883,16.124,-0.679
31,R022_C014,Kilifi,295,-0.324,0.339,-14.651,8.176,-0.663
29,R021_C014,Kilifi,235,0.936,0.310,-3.141,57.354,0.626
33,R023_C014,Kilifi,315,-1.275,-1.862,-11.978,7.289,0.586
6,R004_C007,Kwale-Mombasa,141,0.337,-0.012,-0.605,2.099,0.349


Saved: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\glorys_cell_analysis_QC_verified.csv


In [6]:
from pathlib import Path

# ------------------------------------------------------------
# REPOSITORY ROOT: first folder above this notebook that contains Master_raw
# (e.g. D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024)
# ------------------------------------------------------------
_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents) if (p / "Master_raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No Master_raw folder found above {_here}. Open this notebook inside the repository.")
RAW = ROOT / "Master_raw"          # inputs
OUTPUTS = ROOT / "Outputs"         # everything this repository produces
FIGURES = OUTPUTS / "Figures"      # manuscript figures
TABLES = OUTPUTS / "Tables"        # manuscript tables
TEXT_STATS = TABLES / "Text_statistics"
for _d in (FIGURES, TABLES, TEXT_STATS):
    _d.mkdir(parents=True, exist_ok=True)
OVERWRITE = True                   # re-running replaces this notebook's earlier outputs


def first_existing(*candidates):
    """Return the first path that exists (otherwise the first one)."""
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return Path(candidates[0])


def save_table(df, name, index=False, folder=None):
    path = (folder or TABLES) / f"{name}.csv"
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("Saved:", path.relative_to(ROOT))


def save_figure(fig, name, dpi=600):
    for ext in ("jpg",):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=dpi, bbox_inches="tight", pil_kwargs=({"quality": 95} if ext == "jpg" else None))
    print("Saved:", (FIGURES / f"{name}.jpg").relative_to(ROOT))


def remove_existing(*files):
    for f in map(Path, files):
        for t in (list(f.parent.glob(f.stem + ".*")) if f.suffix == ".shp" else [f]):
            if t.exists():
                if not OVERWRITE:
                    raise FileExistsError(t)
                t.unlink()

# ------------------------------------------------------------
# INPUTS AND OUTPUT FOLDERS FOR THIS NOTEBOOK
# ------------------------------------------------------------
TRANSECT_RATES_DIR = RAW / "Transect_rates"
DSAS_RATES_GPKG = TRANSECT_RATES_DIR / "ke_30mtr_transects_dsas_uncy.gpkg"
DSAS_RATES_LAYER = "ke_30mtr_transect_dsas_uncy_final_rates_20260915_112207"
GLORYS_DIR = OUTPUTS / "GLORYS_attribution"
GLORYS_MIDPOINT_GPKG = GLORYS_DIR / "ke_transects_GLORYS_midpoint_attributed.gpkg"
COMPLETE_LAYER = "transects_complete"
COMPLETE_GPKG = first_existing(GLORYS_DIR / "ke_transects_GLORYS_complete_attribution_zoned.gpkg",
                               TRANSECT_RATES_DIR / "ke_transects_GLORYS_complete_attribution_zoned.gpkg")
COMPLETE_CSV = GLORYS_DIR / "ke_transects_GLORYS_complete_analysis_input_zoned.csv"
GEOSTATS_DIR = OUTPUTS / "Geostatistics"
GEOSTATS_TRANSECT_DIR = GEOSTATS_DIR / "Transect_Level"
GEOSTATS_CELL_DIR = GEOSTATS_DIR / "GLORYS_Cell_Level"

# The QC table is the stage 06 CSV. If only the GeoPackage is available
# (e.g. the copy in Master_raw/Transect_rates), write the same table from it.
if not COMPLETE_CSV.exists() and COMPLETE_GPKG.exists():
    import geopandas as _gpd
    _g = _gpd.read_file(COMPLETE_GPKG, layer=COMPLETE_LAYER)
    COMPLETE_CSV.parent.mkdir(parents=True, exist_ok=True)
    _g.drop(columns="geometry").to_csv(COMPLETE_CSV, index=False, encoding="utf-8-sig", na_rep="")
    print("Built", COMPLETE_CSV.relative_to(ROOT), "from", COMPLETE_GPKG.relative_to(ROOT))

print("Repository root:", ROOT)
for _label, _p in [("COMPLETE_CSV", COMPLETE_CSV), ("DSAS_RATES_GPKG", DSAS_RATES_GPKG), ("GLORYS_MIDPOINT_GPKG", GLORYS_MIDPOINT_GPKG)]:
    print(f"  {'ok     ' if _p.exists() else 'MISSING'} {_label}: {_p.relative_to(ROOT)}")

# ============================================================
# STAGE 6.2: PREPARE AND INSPECT ANALYTICAL GLORYS CELLS
# Join verified cell statistics to native grid polygons
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

GRID_GPKG = GLORYS_MIDPOINT_GPKG

CELL_CSV = GEOSTATS_CELL_DIR / "glorys_cell_analysis_QC_verified.csv"

OUTPUT_DIR = CELL_CSV.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Read native grid and verified analytical data
# ------------------------------------------------------------

grid = gpd.read_file(
    GRID_GPKG,
    layer="glorys_native_grid"
)

cells = pd.read_csv(CELL_CSV)

grid["Pixel_ID"] = grid["Pixel_ID"].astype(str)
cells["Pixel_ID"] = cells["Pixel_ID"].astype(str)

assert grid["Pixel_ID"].is_unique
assert cells["Pixel_ID"].is_unique
assert len(cells) == 47

# ------------------------------------------------------------
# 2. Retain only the 47 analytical grid polygons
# ------------------------------------------------------------

analysis_grid = grid.merge(
    cells.drop(
        columns=[
            "Longitude",
            "Latitude",
            "SL_Trend_mm_yr",
            "SL_Variability_m"
        ],
        errors="ignore"
    ),
    on="Pixel_ID",
    how="inner",
    validate="one_to_one"
)

assert len(analysis_grid) == 47
assert analysis_grid.geometry.notna().all()
assert not analysis_grid.geometry.is_empty.any()

# ------------------------------------------------------------
# 3. Examine polygon geometry and coordinate coverage
# ------------------------------------------------------------

print("TABLE 1 | ANALYTICAL GRID VERIFICATION")

print("Analytical cells:", len(analysis_grid))
print("CRS:", analysis_grid.crs)
print(
    "Geometry types:",
    analysis_grid.geom_type.unique().tolist()
)
print(
    "Invalid geometries:",
    (~analysis_grid.geometry.is_valid).sum()
)

print("\nTABLE 2 | GRID COORDINATE COVERAGE")

display(
    analysis_grid[
        ["Grid_Lon", "Grid_Lat"]
    ].describe().round(4)
)

# ------------------------------------------------------------
# 4. Inspect polygon contiguity
#
# Queen contiguity: polygons share at least one boundary
# point. This is a diagnostic, not a selected weights matrix.
# ------------------------------------------------------------

ids = analysis_grid["Pixel_ID"].tolist()
geometries = analysis_grid.geometry.tolist()

neighbours = {}

for i, pixel_id in enumerate(ids):

    neighbours[pixel_id] = []

    for j, other_id in enumerate(ids):

        if i == j:
            continue

        if geometries[i].touches(geometries[j]):
            neighbours[pixel_id].append(other_id)

analysis_grid["Queen_Neighbour_Count"] = (
    analysis_grid["Pixel_ID"]
    .map(lambda x: len(neighbours[x]))
)

print("\nTABLE 3 | POLYGON CONTIGUITY")

display(
    analysis_grid[
        "Queen_Neighbour_Count"
    ].describe()
)

isolated = analysis_grid.loc[
    analysis_grid["Queen_Neighbour_Count"] == 0,
    [
        "Pixel_ID",
        "Grid_Row",
        "Grid_Col",
        "Grid_Lon",
        "Grid_Lat",
        "Zones",
        "Valid_WLR_Count"
    ]
]

print("\nCells without touching analytical neighbours:")
print(len(isolated))

display(isolated)

# ------------------------------------------------------------
# 5. Save geospatial analysis dataset
# ------------------------------------------------------------

OUTPUT_GPKG = (
    OUTPUT_DIR /
    "glorys_47_cells_QC_verified.gpkg"
)

remove_existing(OUTPUT_GPKG)
analysis_grid.to_file(
    OUTPUT_GPKG,
    layer="glorys_47_cells",
    driver="GPKG"
)

print("\nSaved:", OUTPUT_GPKG)



Repository root: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024
  ok      COMPLETE_CSV: Outputs\GLORYS_attribution\ke_transects_GLORYS_complete_analysis_input_zoned.csv
  ok      DSAS_RATES_GPKG: Master_raw\Transect_rates\ke_30mtr_transects_dsas_uncy.gpkg
  ok      GLORYS_MIDPOINT_GPKG: Outputs\GLORYS_attribution\ke_transects_GLORYS_midpoint_attributed.gpkg
TABLE 1 | ANALYTICAL GRID VERIFICATION
Analytical cells: 47
CRS: EPSG:4326
Geometry types: ['Polygon']
Invalid geometries: 0

TABLE 2 | GRID COORDINATE COVERAGE


,Grid_Lon,Grid_Lat
count,47.0000,47.0000
mean,40.0248,-3.4344
std,0.4004,0.7240
min,39.2500,-4.6667
25%,39.7500,-4.0417
50%,40.0000,-3.3333
75%,40.2500,-2.7500
max,40.8333,-2.4167



TABLE 3 | POLYGON CONTIGUITY


count    47.000000
mean      3.106383
std       0.729325
min       1.000000
25%       3.000000
50%       3.000000
75%       4.000000
max       4.000000
Name: Queen_Neighbour_Count, dtype: float64


Cells without touching analytical neighbours:
0


,Pixel_ID,Grid_Row,Grid_Col,Grid_Lon,Grid_Lat,Zones,Valid_WLR_Count



Saved: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\glorys_47_cells_QC_verified.gpkg


In [7]:
# ============================================================
# STAGE 7: QUEEN-WEIGHTS NETWORK CONNECTIVITY
# Diagnostic only: no spatial weights selected yet
# ============================================================

import pandas as pd

# 1. Check that all analytical cells are represented
ids = set(analysis_grid["Pixel_ID"].astype(str))

assert set(neighbours.keys()) == ids

# 2. Find connected components using graph traversal
unvisited = set(ids)
components = []

while unvisited:

    start = unvisited.pop()
    component = {start}
    stack = [start]

    while stack:

        current = stack.pop()

        for neighbour in neighbours[current]:

            if neighbour in unvisited:

                unvisited.remove(neighbour)
                component.add(neighbour)
                stack.append(neighbour)

    components.append(sorted(component))

# 3. Summarise connected components
component_summary = pd.DataFrame({
    "Component": range(1, len(components) + 1),
    "Number_of_Cells": [len(c) for c in components],
    "Pixel_IDs": [", ".join(c) for c in components]
})

print("TABLE 4 | QUEEN NETWORK CONNECTIVITY")

print("Number of connected components:", len(components))
print("Largest component:", max(map(len, components)))
print("Smallest component:", min(map(len, components)))

display(component_summary)

# 4. Verify neighbour relationships are reciprocal
asymmetric = []

for pixel_id, adjacent in neighbours.items():

    for other_id in adjacent:

        if pixel_id not in neighbours[other_id]:
            asymmetric.append((pixel_id, other_id))

print("\nAsymmetric neighbour relationships:", len(asymmetric))

# 5. Interpretation
if len(components) == 1 and not asymmetric:

    print(
        "\nRESULT: All 47 analytical cells form one "
        "connected, reciprocal Queen-neighbour network."
    )

else:

    print(
        "\nRESULT: The network requires further inspection "
        "before spatial weights are selected."
    )

TABLE 4 | QUEEN NETWORK CONNECTIVITY
Number of connected components: 1
Largest component: 47
Smallest component: 47


,Component,Number_of_Cells,Pixel_IDs
0,1,47,"R001_C003, R001_C004, R001_C005, R002_C005, R0..."



Asymmetric neighbour relationships: 0

RESULT: All 47 analytical cells form one connected, reciprocal Queen-neighbour network.


In [8]:
# ============================================================
# STAGE 8: AUDIT GLORYS QUEEN-NEIGHBOUR CONNECTIONS
# Diagnostic only; no weights selected
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Geod

# Use the 47-cell GeoDataFrame from Stage 6.
assert len(analysis_grid) == 47
assert analysis_grid["Pixel_ID"].is_unique

grid_lookup = analysis_grid.set_index("Pixel_ID")
geod = Geod(ellps="WGS84")

records = []
seen = set()

for pixel_id, adjacent in neighbours.items():
    for other_id in adjacent:

        pair = tuple(sorted((pixel_id, other_id)))

        if pair in seen:
            continue

        seen.add(pair)

        a = grid_lookup.loc[pair[0]]
        b = grid_lookup.loc[pair[1]]

        shared_boundary = (
            a.geometry.boundary.intersection(
                b.geometry.boundary
            )
        )

        # A shared line indicates edge adjacency.
        # Point-only intersection indicates corner adjacency.
        has_shared_edge = shared_boundary.length > 1e-10

        connection_type = (
            "Shared edge"
            if has_shared_edge
            else "Corner only"
        )

        _, _, distance_m = geod.inv(
            float(a["Grid_Lon"]),
            float(a["Grid_Lat"]),
            float(b["Grid_Lon"]),
            float(b["Grid_Lat"])
        )

        records.append({
            "Pixel_A": pair[0],
            "Pixel_B": pair[1],
            "Connection_Type": connection_type,
            "Distance_km": distance_m / 1000,
            "Zone_A": a["Zones"],
            "Zone_B": b["Zones"],
            "Grid_Row_A": a["Grid_Row"],
            "Grid_Col_A": a["Grid_Col"],
            "Grid_Row_B": b["Grid_Row"],
            "Grid_Col_B": b["Grid_Col"]
        })

edge_audit = pd.DataFrame(records)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(edge_audit) > 0
assert not edge_audit[
    ["Pixel_A", "Pixel_B"]
].duplicated().any()

expected_pairs = (
    analysis_grid["Queen_Neighbour_Count"].sum() / 2
)

assert len(edge_audit) == expected_pairs

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("TABLE 5 | NEIGHBOUR CONNECTION SUMMARY")

print("Unique connections:", len(edge_audit))

display(
    edge_audit["Connection_Type"]
    .value_counts()
    .rename_axis("Connection_Type")
    .reset_index(name="Connections")
)

print("\nTABLE 6 | CONNECTION DISTANCES (km)")

display(
    edge_audit.groupby("Connection_Type")[
        "Distance_km"
    ].describe().round(3)
)

print("\nTABLE 7 | LONGEST CONNECTIONS")

display(
    edge_audit.sort_values(
        "Distance_km",
        ascending=False
    ).head(15).round(3)
)

# ------------------------------------------------------------
# Export audit
# ------------------------------------------------------------

AUDIT_FILE = (
    OUTPUT_DIR /
    "glorys_queen_neighbour_audit.csv"
)

edge_audit.to_csv(
    AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", AUDIT_FILE)

TABLE 5 | NEIGHBOUR CONNECTION SUMMARY
Unique connections: 73


,Connection_Type,Connections
0,Shared edge,45
1,Corner only,28



TABLE 6 | CONNECTION DISTANCES (km)


,count,mean,std,min,25%,50%,75%,max
Connection_Type,,,,,,,,
Corner only,28.0,13.064,0.005,13.055,13.060,13.065,13.068,13.069
Shared edge,45.0,9.234,0.023,9.215,9.215,9.215,9.261,9.268



TABLE 7 | LONGEST CONNECTIONS


,Pixel_A,Pixel_B,Connection_Type,Distance_km,Zone_A,Zone_B,Grid_Row_A,Grid_Col_A,Grid_Row_B,Grid_Col_B
69,R027_C020,R028_C021,Corner only,13.069,Tana-Manda,Tana-Manda,27,20,28,21
71,R027_C021,R028_C022,Corner only,13.069,Tana-Manda,Tana-Manda,27,21,28,22
65,R026_C019,R027_C020,Corner only,13.069,Tana-Manda,Tana-Manda,26,19,27,20
67,R026_C020,R027_C021,Corner only,13.069,Tana-Manda,Tana-Manda,26,20,27,21
60,R025_C016,R026_C017,Corner only,13.069,Tana-Manda,Tana-Manda,25,16,26,17
58,R025_C015,R026_C016,Corner only,13.068,Tana-Manda,Tana-Manda,25,15,26,16
56,R024_C015,R025_C016,Corner only,13.068,Kilifi | Tana-Manda,Tana-Manda,24,15,25,16
54,R024_C014,R025_C015,Corner only,13.068,Kilifi,Tana-Manda,24,14,25,15
52,R023_C014,R024_C015,Corner only,13.068,Kilifi,Kilifi | Tana-Manda,23,14,24,15
50,R022_C015,R023_C014,Corner only,13.067,Kilifi,Kilifi,22,15,23,14



Saved: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\glorys_queen_neighbour_audit.csv


In [9]:
# ============================================================
# STAGE 9: COMPARE QUEEN AND ROOK NETWORK CONNECTIVITY
# Diagnostic only; no weights selected
# ============================================================

import pandas as pd

ids = analysis_grid["Pixel_ID"].astype(str).tolist()

# ------------------------------------------------------------
# 1. Construct Rook neighbours from verified edge audit
# ------------------------------------------------------------

rook_neighbours = {pixel_id: [] for pixel_id in ids}

rook_edges = edge_audit.loc[
    edge_audit["Connection_Type"] == "Shared edge"
]

for _, edge in rook_edges.iterrows():

    a = str(edge["Pixel_A"])
    b = str(edge["Pixel_B"])

    rook_neighbours[a].append(b)
    rook_neighbours[b].append(a)

# ------------------------------------------------------------
# 2. Reusable connected-component function
# ------------------------------------------------------------

def connected_components(neighbour_dict):

    unvisited = set(neighbour_dict)
    components = []

    while unvisited:

        start = unvisited.pop()
        component = {start}
        stack = [start]

        while stack:

            current = stack.pop()

            for neighbour in neighbour_dict[current]:

                if neighbour in unvisited:

                    unvisited.remove(neighbour)
                    component.add(neighbour)
                    stack.append(neighbour)

        components.append(sorted(component))

    return sorted(
        components,
        key=len,
        reverse=True
    )

# ------------------------------------------------------------
# 3. Compare network structure
# ------------------------------------------------------------

queen_components = connected_components(neighbours)
rook_components = connected_components(rook_neighbours)

summary = pd.DataFrame({
    "Diagnostic": [
        "Cells",
        "Unique connections",
        "Connected components",
        "Largest component",
        "Isolated cells",
        "Minimum neighbours",
        "Mean neighbours",
        "Maximum neighbours"
    ],
    "Queen": [
        len(ids),
        len(edge_audit),
        len(queen_components),
        len(queen_components[0]),
        sum(len(v) == 0 for v in neighbours.values()),
        min(map(len, neighbours.values())),
        sum(map(len, neighbours.values())) / len(ids),
        max(map(len, neighbours.values()))
    ],
    "Rook": [
        len(ids),
        len(rook_edges),
        len(rook_components),
        len(rook_components[0]),
        sum(len(v) == 0 for v in rook_neighbours.values()),
        min(map(len, rook_neighbours.values())),
        sum(map(len, rook_neighbours.values())) / len(ids),
        max(map(len, rook_neighbours.values()))
    ]
})

print("TABLE 8 | QUEEN VS ROOK CONNECTIVITY")
display(summary)

# ------------------------------------------------------------
# 4. Identify disconnected Rook components
# ------------------------------------------------------------

print("\nTABLE 9 | ROOK CONNECTED COMPONENTS")

component_table = pd.DataFrame({
    "Component": range(1, len(rook_components) + 1),
    "Number_of_Cells": [
        len(component)
        for component in rook_components
    ],
    "Pixel_IDs": [
        ", ".join(component)
        for component in rook_components
    ]
})

display(component_table)

# ------------------------------------------------------------
# 5. Identify cells with no Rook neighbours
# ------------------------------------------------------------

rook_isolated = [
    pixel_id
    for pixel_id, adjacent in rook_neighbours.items()
    if len(adjacent) == 0
]

print("\nROOK ISOLATED CELLS")
print(rook_isolated)

# ------------------------------------------------------------
# 6. Save diagnostic summary
# ------------------------------------------------------------

summary.to_csv(
    OUTPUT_DIR / "glorys_queen_rook_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

component_table.to_csv(
    OUTPUT_DIR / "glorys_rook_components.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nStage 9 completed.")

TABLE 8 | QUEEN VS ROOK CONNECTIVITY


,Diagnostic,Queen,Rook
0,Cells,47.000000,47.000000
1,Unique connections,73.000000,45.000000
2,Connected components,1.000000,3.000000
3,Largest component,47.000000,38.000000
4,Isolated cells,0.000000,0.000000
5,Minimum neighbours,1.000000,1.000000
6,Mean neighbours,3.106383,1.914894
7,Maximum neighbours,4.000000,3.000000



TABLE 9 | ROOK CONNECTED COMPONENTS


,Component,Number_of_Cells,Pixel_IDs
0,1,38,"R007_C008, R008_C008, R008_C009, R009_C009, R0..."
1,2,5,"R003_C006, R004_C006, R004_C007, R005_C007, R0..."
2,3,4,"R001_C003, R001_C004, R001_C005, R002_C005"



ROOK ISOLATED CELLS
[]

Stage 9 completed.


In [10]:
# ============================================================
# STAGE 10: IDENTIFY CONNECTIONS BETWEEN ROOK COMPONENTS
# Diagnostic only; no spatial weights selected
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Assign each cell to its Rook component
# ------------------------------------------------------------

component_lookup = {}

for component_number, component in enumerate(
    rook_components, start=1
):
    for pixel_id in component:
        component_lookup[pixel_id] = component_number

assert len(component_lookup) == 47

# ------------------------------------------------------------
# 2. Add component membership to Queen edge audit
# ------------------------------------------------------------

connection_audit = edge_audit.copy()

connection_audit["Component_A"] = (
    connection_audit["Pixel_A"]
    .map(component_lookup)
)

connection_audit["Component_B"] = (
    connection_audit["Pixel_B"]
    .map(component_lookup)
)

if connection_audit[
    ["Component_A", "Component_B"]
].isna().any().any():
    raise ValueError("Missing Rook component assignment.")

# ------------------------------------------------------------
# 3. Identify connections between components
# ------------------------------------------------------------

bridges = connection_audit.loc[
    connection_audit["Component_A"]
    != connection_audit["Component_B"]
].copy()

bridges = bridges.sort_values(
    ["Component_A", "Component_B", "Pixel_A"]
)

print("TABLE 10 | CONNECTIONS BETWEEN ROOK COMPONENTS")

print("Number of cross-component connections:", len(bridges))

display(
    bridges[
        [
            "Pixel_A",
            "Pixel_B",
            "Connection_Type",
            "Distance_km",
            "Zone_A",
            "Zone_B",
            "Component_A",
            "Component_B"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 4. Summarise component connections
# ------------------------------------------------------------

bridges["Component_Pair"] = bridges.apply(
    lambda row: " - ".join(
        map(
            str,
            sorted([
                int(row["Component_A"]),
                int(row["Component_B"])
            ])
        )
    ),
    axis=1
)

print("\nTABLE 11 | COMPONENT CONNECTION SUMMARY")

display(
    bridges.groupby("Component_Pair")
    .agg(
        Connections=("Pixel_A", "size"),
        Mean_Distance_km=("Distance_km", "mean"),
        Maximum_Distance_km=("Distance_km", "max")
    )
    .reset_index()
    .round(3)
)

# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

OUTPUT_FILE = (
    OUTPUT_DIR /
    "glorys_rook_component_bridge_audit.csv"
)

bridges.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", OUTPUT_FILE)

TABLE 10 | CONNECTIONS BETWEEN ROOK COMPONENTS
Number of cross-component connections: 2


,Pixel_A,Pixel_B,Connection_Type,Distance_km,Zone_A,Zone_B,Component_A,Component_B
11,R006_C007,R007_C008,Corner only,13.058,Kwale-Mombasa,Kwale-Mombasa,2,1
4,R002_C005,R003_C006,Corner only,13.055,Kwale-Mombasa,Kwale-Mombasa,3,2



TABLE 11 | COMPONENT CONNECTION SUMMARY


,Component_Pair,Connections,Mean_Distance_km,Maximum_Distance_km
0,1 - 2,1,13.058,13.058
1,2 - 3,1,13.055,13.055



Saved: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\glorys_rook_component_bridge_audit.csv


In [11]:
# ============================================================
# STAGE 13: PREPARE QUEEN AND ROOK SPATIAL WEIGHTS
# Validation and comparison only
# ============================================================

import pandas as pd
import numpy as np

# Check objects from previous stages.
required = [
    "analysis_grid",
    "neighbours",
    "rook_neighbours"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing objects: " + ", ".join(missing)
    )

# Establish one consistent ordering for both matrices.
cell_ids = analysis_grid["Pixel_ID"].astype(str).tolist()

if len(cell_ids) != len(set(cell_ids)):
    raise ValueError("Duplicate Pixel_ID values detected.")

n = len(cell_ids)
id_to_index = {
    cell_id: i
    for i, cell_id in enumerate(cell_ids)
}

def build_weights(neighbour_dict, name):

    if set(neighbour_dict) != set(cell_ids):
        raise ValueError(
            f"{name}: neighbour IDs do not match analysis cells."
        )

    W = np.zeros((n, n), dtype=float)

    for cell_id, adjacent in neighbour_dict.items():

        i = id_to_index[cell_id]

        for other_id in adjacent:

            if other_id not in id_to_index:
                raise ValueError(
                    f"{name}: unknown neighbour {other_id}"
                )

            j = id_to_index[other_id]

            if i == j:
                raise ValueError(
                    f"{name}: self-neighbour detected."
                )

            W[i, j] = 1.0

    # Check symmetry before standardisation.
    if not np.array_equal(W, W.T):
        raise ValueError(
            f"{name}: asymmetric neighbour relationships."
        )

    counts = W.sum(axis=1)

    if np.any(counts == 0):
        raise ValueError(
            f"{name}: isolated cells detected."
        )

    # Row standardisation.
    W_row = W / counts[:, None]

    return W, W_row, counts

queen_binary, queen_W, queen_counts = build_weights(
    neighbours,
    "Queen"
)

rook_binary, rook_W, rook_counts = build_weights(
    rook_neighbours,
    "Rook"
)

comparison = pd.DataFrame({
    "Weights": ["Queen", "Rook"],
    "Cells": [n, n],
    "Undirected_Edges": [
        int(queen_binary.sum() / 2),
        int(rook_binary.sum() / 2)
    ],
    "Minimum_Neighbours": [
        int(queen_counts.min()),
        int(rook_counts.min())
    ],
    "Mean_Neighbours": [
        queen_counts.mean(),
        rook_counts.mean()
    ],
    "Maximum_Neighbours": [
        int(queen_counts.max()),
        int(rook_counts.max())
    ],
    "Isolates": [
        int((queen_counts == 0).sum()),
        int((rook_counts == 0).sum())
    ]
})

print("TABLE 16 | SPATIAL WEIGHTS COMPARISON")
display(comparison.round(3))

print("\nMatrix validation:")
print("Queen row sums:", np.unique(queen_W.sum(axis=1)))
print("Rook row sums:", np.unique(rook_W.sum(axis=1)))

print("\nStage 13 completed.")

# Persist analysis order and both binary and row-standardised matrices.
# Index/column labels are essential: never infer row order from a CSV alone.
ORDER_FILE = CELL_DIR / "glorys_47_cell_order.csv"
pd.DataFrame({"Pixel_ID": cell_ids, "Matrix_Row": np.arange(n)}).to_csv(ORDER_FILE,index=False)
for label, binary, standardised in [("queen",queen_binary,queen_W),("rook",rook_binary,rook_W)]:
    pd.DataFrame(binary,index=cell_ids,columns=cell_ids).to_csv(CELL_DIR/f"glorys_{label}_weights_binary.csv")
    pd.DataFrame(standardised,index=cell_ids,columns=cell_ids).to_csv(CELL_DIR/f"glorys_{label}_weights_row_standardised.csv")
assert n == 47 and int(queen_binary.sum()/2)==73 and int(rook_binary.sum()/2)==45
assert np.allclose(queen_W.sum(axis=1),1) and np.allclose(rook_W.sum(axis=1),1)
print("Exported labelled Queen/Rook matrices and 47-cell order to", CELL_DIR)


TABLE 16 | SPATIAL WEIGHTS COMPARISON


,Weights,Cells,Undirected_Edges,Minimum_Neighbours,Mean_Neighbours,Maximum_Neighbours,Isolates
0,Queen,47,73,1,3.106,4,0
1,Rook,47,45,1,1.915,3,0



Matrix validation:
Queen row sums: [1.]
Rook row sums: [1.]

Stage 13 completed.
Exported labelled Queen/Rook matrices and 47-cell order to D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level
